## RL Methods: Dynamic Programming (DP)

- The simplest case of solving the Bellman Equation is when you have **both** the transition probabilities $P(s'|s,a)$ and the rewards $R(s,a,s')$

- That is, you know the full Markov Decision Process (MDP). Hence the term "model based", because we are using the MDP model

- This is, of course, exceedingly rare. But it is instructive to see how to solve for $\pi^*$ when you have almost all available information

- We will go through 2 ways of implementing the DP solver
    - Policy Interation
    - Value Iteration

- To do this, let's assume the following set up:

In [ ]:
import numpy as np

np.random.seed(123)

N_STATES = 20
N_ACTIONS = 10
GAMMA = 0.9

# Shape corresponding to State, Action, Next State
TRANSITION_PROBABILITIES = np.random.rand(N_STATES, N_ACTIONS, N_STATES)

# Normalise. We want to make sure that for the same start state (each block), the same action taken (each row),
# the probability of ending up at each end state (each column) sums to 1
TRANSITION_PROBABILITIES /= np.sum(TRANSITION_PROBABILITIES, axis=2, keepdims=True)

REWARDS = np.random.rand(N_STATES, N_ACTIONS, N_STATES) * 5

In [ ]:
TRANSITION_PROBABILITIES.shape, REWARDS.shape

### Policy Iteration

- In policy iteration, we create a loop that does (i) policy evaluation, and (ii) policy improvement based on the evaluated outcomes of the policy, and the loop runs until some stopping criteria is reached

- In the DP RL setting, there are 4 factors that affect the overall decision. 2 will come from the environment, and 2 will come from the agent:
    - Environment will dictate:
        - Transition probability $P$: Given a state $S$ and an action $A$, what is the probability of transitioning to some new state $S'$
        - Reward $R$: For any given `(state, action, new state)`, how much reward do we experience?
    - Agent will learn:
        - Policy $\pi$: At a given state $S$, what action $A$ should be taken?
        - State Value Function $V(s)$: What is the expected long term reward $V(s)$ of starting from state $s$ following policy $\pi$
        - Action value function $Q(s,a)$: What is the expected return starting from state $s$, taking action $a$, and then following policy $\pi$.

- **Policy Evaluation:** 
    - The main purpose of this step is to use what we know about transition probabilities and rewards to update our state value function. 
    
    - We **DO NOT** change our decision making policy here; the question we answer is: given our decision making policy, the transition probability, and rewards of the environment, how should we update our expectation of the environment's state value expectations

    - Since we know the transition probabilities and rewards with full certainty in DP, we can update the State-Value function by computing the Bellman Equation
    
    - Start with a seed array representing the State-Value function $V$, and a policy $\pi$ that deterministically tells us what action to take in a given state $S$
    
    - Compute 1 pass of the Bellman State Value function update for all states. This is just a transition-probability-weighted average of the state values of the next states $S'$

    - Check that the largest change in the state value estimates for all $N$ states exceeds some threshold $\theta$. If even the largest change falls below the threshold, we assume convergence, and we return the state values $V$

    - Otherwise, keep looping

- **Policy Improvement**
    - We've previously updated our estimate of the State-Value function $V$

    - Now, we answer a different question. Given our updated environment state values, the transition probability, and rewards of the environment, how do we update our decision making policy to take better actions at every given state.

    - Now, we need to update our policy based on the new estimate of the State-Value function. That is, given a state $S$, I want to know the value of taking an action $A$. 

    - That is; the **Q-Value**

    - Looping over every state $S$, we init an array to hold the new Q values

    - Given $S$, loop over every action available, and update the Action-Value function $Q$. This is simply the transition-probability-weighted average of rewards from the transition $S -> S'$, plus the discounted state-value of $S'$

    - Finally, the new policy for state $S$ is simply to take the argmax of all the Q values

- **Policy Iteration**
    - Bringing these 2 together, we simply create an infinite loop of `Evaluation --> Improvement --> Evalution ...`

    - This will keep going until some convergence criteria is met. An easy criteria might be to say that; if for some $N$ consecutive loops we do not change our policy, we have reached convergence

In [ ]:
def policy_evaluation(
    transition_probabilities: np.ndarray,
    rewards: np.ndarray,
    curr_state_value_function: np.ndarray,
    policy: np.ndarray,
    gamma: float = 0.9,
    theta: float = 1e-6
) -> np.ndarray:
    '''
    What is the long term value of state S if I follow policy $\pi$, given transition probabilities $P$ and one-step rewards $R$?
    '''
    nstates = transition_probabilities.shape[0]
    nactions = transition_probabilities.shape[1]
    updated_state_value_function = curr_state_value_function.copy()
    while True:
        max_improvement = 0
        for curr_state in range(nstates):
            v_updated = sum([
                transition_probabilities[curr_state, policy[curr_state], s_prime] * 
                (rewards[curr_state, policy[curr_state], s_prime] + gamma * updated_state_value_function[s_prime])
                for s_prime in range(nstates)
            ])
            max_improvement = max(max_improvement, abs(v_updated - updated_state_value_function[curr_state]))
            updated_state_value_function[curr_state] = v_updated
        if max_improvement < theta:
            break
    
    return updated_state_value_function

def policy_improvement(
    transition_probabilities: np.ndarray,
    rewards: np.ndarray,
    curr_state_value_function: np.ndarray,
    curr_action_value_function: np.ndarray,
    curr_policy: np.ndarray,
    gamma: float = 0.9
) -> tuple[np.ndarray, np.ndarray]:
    '''
    With my new estimate of long term value for a given state $S$, what new policy $\pi$ should dictate my actions, 
    given transition probabilities $P$ and one-step rewards $R$?
    '''
    nstates = transition_probabilities.shape[0]
    nactions = transition_probabilities.shape[1]
    updated_policy = curr_policy.copy()
    updated_action_value_function = curr_action_value_function.copy()
    for s in range(nstates):
        action_value_function_for_state_s = updated_action_value_function[s].copy()
        for a in range(nactions):
            new_action_value = sum([
                transition_probabilities[s, a, s_prime] * 
                (rewards[s, a, s_prime] + gamma * curr_state_value_function[s_prime])
                for s_prime in range(nstates)
            ])
            action_value_function_for_state_s[a] = new_action_value
        
        updated_policy[s] = np.argmax(action_value_function_for_state_s)
        updated_action_value_function[s] = action_value_function_for_state_s
    return updated_policy, updated_action_value_function

def policy_iteration(
    transition_probabilities: np.ndarray,
    rewards: np.ndarray,
    gamma: float = 0.9,
    theta: float = 1e-6
):
    nstates = transition_probabilities.shape[0]
    nactions = transition_probabilities.shape[1]
    curr_state_value_function: np.ndarray = np.zeros(nstates)
    curr_action_value_function: np.ndarray = np.zeros((nstates, nactions))
    curr_policy: np.ndarray = np.zeros(nstates).astype(int)
    
    count_no_change = 0
    count_iters = 0
    while True:
        updated_state_value_function = policy_evaluation(
            transition_probabilities,
            rewards,
            curr_state_value_function,
            curr_policy,
            gamma,
            theta
        )
        updated_policy, updated_action_value_function = policy_improvement(
            transition_probabilities,
            rewards,
            updated_state_value_function,
            curr_action_value_function,
            curr_policy,
            gamma
        )
        if np.array_equal(curr_policy, updated_policy):
            count_no_change += 1
            if count_no_change >= 5:
                break
        else:
            count_no_change = 0

        curr_policy = updated_policy.copy()
        curr_action_value_function = updated_action_value_function.copy()
        curr_state_value_function = updated_state_value_function.copy()

        count_iters += 1
    
    return curr_policy, curr_action_value_function, curr_state_value_function

pi_star, q_star, v_star = policy_iteration(TRANSITION_PROBABILITIES, REWARDS)

print(f"""
    Optimal Policy: {pi_star}
    Action-Value: {q_star}
    State-Value: {v_star}
""")

- To ascertain if your policy iteration has converged correctly, check that the action values computed from the current transition probability, rewards, and optimal state values matches the values in your optimal q value array 

In [ ]:
for s in range(N_STATES):
    for a in range(N_ACTIONS):
        q = sum([
            TRANSITION_PROBABILITIES[s, a, s_prime] *
            (REWARDS[s, a, s_prime] + GAMMA * v_star[s_prime])
            for s_prime in range(N_STATES)
        ])
        assert np.isclose(q, q_star[s, a])

### Value Iteration

- In policy iteration, notice we have 2 distinct steps; policy evaluation, then improvement

- The key difference between policy iteration and value iteraton is in how long we allow the policy evaluation to occur before we search for a policy improvement:
    
    - For policy iteration:
        - Perform policy evaluation exhaustively. That is, we use policy $\pi$ to update the state value function $v$ until convergence
        - Only after we reach some convergence in the estimate of state value, then we perform policy improvement. We now use the "converged" state value function $v$ to pick the best action $a$ for every state $s$

    - For value iteration:
        - We still perform policy evaluation. But instead of iteratively updating the value function until convergence, we don't wait until the value function converges before doing a policy update. 
        - Instead, after a single value function update step, we update our policy immediately, either taking the maximum value across all possible actions at a given state, or some sort of sampling

- While both approaches converge to the same answer, there is typically a difference in speed.
    - Policy Iteration often takes fewer iterations to find the optimal policy, but each iteration is "expensive" because it involves a full convergence loop 
    - Value Iteration takes more iterations, but each iteration is "cheap" and fast
    - PRACTICALLY speaking, we don't actually care about the optimal estimate of value function!! That is, so long as we make the right decision directionally, we don't really care about the actual state value. 
        - In most cases, policy ($\pi$) converges to "optimal" long before the value function ($v$) does
        - This is why policy iteration is generally faster, because training stops when policy becomes stable

In [ ]:
def value_iteration(
    transition_probabilities: np.ndarray,
    rewards: np.ndarray,
    gamma: float = 0.9,
    max_rounds_without_policy_change: int = 25
) -> np.ndarray:
    nstates = transition_probabilities.shape[0]
    nactions = transition_probabilities.shape[1]
    curr_state_value_function = np.zeros(nstates)
    curr_policy = np.zeros(nstates, dtype=int)
    
    count_no_change = 0
    
    while True:
        new_state_value_function = np.zeros(nstates)
        new_policy = np.zeros(nstates, dtype=int)
        
        for s in range(nstates):
            action_values = []
            for a in range(nactions):
                action_value = sum([
                    transition_probabilities[s, a, s_prime] * (rewards[s, a, s_prime] + gamma * curr_state_value_function[s_prime])
                    for s_prime in range(nstates)
                ])
                action_values.append(action_value)
            
            new_state_value_function[s] = max(action_values)
            
            new_policy[s] = np.argmax(action_values)

        if np.array_equal(new_policy, curr_policy):
            count_no_change += 1
        else:
            count_no_change = 0
            
        # Stop if policy stays stable for N rounds
        if count_no_change >= max_rounds_without_policy_change:
            break
            
        curr_state_value_function = new_state_value_function
        curr_policy = new_policy
    
    return curr_policy

In [ ]:
value_iteration(TRANSITION_PROBABILITIES, REWARDS)

### Practical Implementation: The Frozen Lake (Grid World)

- We generated a random transition probability and reward arrays above. Now, let's make a proper environment with a clearly understandable success and failure conditions

- We generate 2 $N \times A \times N$ grids below. 
    - One of them is the transition probability matrix. For simplicity, we simply assume that transition is deterministic. If I take an "up" action, I will go up with 100% certainty
    - The other is a reward matrix. For any action taken, we allocate a -0.1 value to the action. So unnecessaily long steps are penalised. 
        - And we allocate the reward of 10 from landing on the target point

- We will use the value and policy iteration approaches above to solve for the policy we should take

In [ ]:
def make_gridworld(size=4):
    n_s = size * size # N x N grid
    n_a = 5  # 0: Up, 1: Down, 2: Left, 3: Right, 4: Stay
    
    P = np.zeros((n_s, n_a, n_s)) # Transition probabilities: If I'm in state S, and take action A, what is the probability I transition to S'
    R = np.zeros((n_s, n_a, n_s)) # Transition probabilities: If I'm in state S, and take action A, what reward do I get?
    
    for s in range(n_s):
        # Looping through every index in the grid
        row_index, col_index = divmod(s, size)
        
        # For every possible action, conditioning on your starting position `s`, what is the next cell you land in if you take action `a`. Recall, we enforce 
        # deterministic movement by assumption
        for a in range(n_a):
            if a == 0: # Up
                next_row, next_col = max(row_index - 1, 0), col_index
            elif a == 1: # Down
                next_row, next_col = min(row_index + 1, size - 1), col_index
            elif a == 2: # Left
                next_row, next_col = row_index, max(col_index - 1, 0)
            elif a == 3: # Right
                next_row, next_col = row_index, min(col_index + 1, size - 1)
            elif a == 4: # Stay
                next_row, next_col = row_index, col_index
            
            s_prime = next_row * size + next_col
            
            # 100% probability to move to intended cell
            P[s, a, s_prime] = 1.0
            
            # Each move costs you something
            R[s, a, s_prime] = -0.1

    # Decide which is your target cell, and give is some big reward       
    target_cell = np.random.randint(1, n_s)
    R[:, :, target_cell] += 10.0
    
    return P, R

SIZE=5
P_grid, R_grid = make_gridworld(size=SIZE)

In [ ]:
def print_policy(policy, size=4):
    ## Print the policy you learn in the N x N grid format
    symbols = ['up', 'down', 'left', 'right', 'stay']
    policy_grid = np.array([symbols[a] for a in policy]).reshape(size, size)
    print(policy_grid)

In [ ]:
pi_star_grid_vi = value_iteration(P_grid, R_grid)
pi_star_grid_pi, q_star_grid_pi, v_star_grid_pi = policy_iteration(P_grid, R_grid)

In [ ]:
print_policy(pi_star_grid_vi, SIZE)
print_policy(pi_star_grid_pi, SIZE)

### Additional: Asynchronous Value Iteration

- Recap:
    - We started by establishing the idea of policy iteration above. Recap: 
        - We fix the policy we currently have, and use it to interact with the fixed environmental processes `transition_probability` and `reward`, and update our estimate of how valuable it is to be in each state $S$
        - Once we converge to a stable expectation of each state $S$ (i.e. once the value function converges), we fix our value function, and update our policy by taking the argmax of the rewards of all state transitions $S --> S'$
        - Repeat until some convergence criteria is met

    - Then, we claimed that this is inefficient, because this forces us to estimate value function precisely, which we actually don't care about. What we really want is to have a policy that is directionally sensible. The precision of the actual value function, whether it is [1,2,3] or [1.1,2.2,3.3], is largely irrelevant, so long as we know how to take the right action

    - To exploit this, we use value iteration instead. Recap:
        - Instead of splitting the policy evaluation and policy updating, we do it all in one pass
        - Given the `transition_probability` and `reward` of the current environment, and our `curr_state_value_function`, we compute the action values of specific actions at a given state $S$
        - We don't compute the state value function explicitly; instead, we compute it implicitly using the max action value function for a given state $S$
        - We expect this to be faster, because we don't bother computing state value function until convergence; instead, we simply update our actions using the current best guess of state value, then update our state value using our current best guess of action value

- Now, as it turns out, we can do EVEN better than value iteration! 
    - Notice how, in value interation, 